In [33]:
import ast
import pandas as pd

education_clean = pd.read_csv("../raw/education_raw.csv")

dups = education_clean.duplicated().sum()
print(f"Duplicate rows: {dups}")

##dups_id = education_clean.duplicated(subset=["countryiso3code"]).sum()
##print(f"Duplicate id: {dups_id}") composit key needed
dups_combo = education_clean.duplicated(subset=["countryiso3code", "indicator_code", "date"]).sum()
print(f"Duplicate (country + indicator + date): {dups_combo}")
print(f"\nShape: {education_clean.shape}")

Duplicate rows: 0
Duplicate (country + indicator + date): 120

Shape: (3000, 9)


In [34]:
print(education_clean.columns.tolist())

['indicator', 'country', 'countryiso3code', 'date', 'value', 'unit', 'obs_status', 'decimal', 'indicator_code']


In [35]:
example = education_clean[education_clean.duplicated(subset=["countryiso3code", "indicator_code", "date"], keep=False)]
print(f"Total rows involved: {len(example)}")
print(example.head(4).to_string())

Total rows involved: 150
                                                                  indicator                               country countryiso3code  date      value  unit  obs_status  decimal indicator_code
160  {'id': 'SE.PRM.ENRR', 'value': 'School enrollment, primary (% gross)'}  {'id': 'XD', 'value': 'High income'}             NaN  2025        NaN   NaN         NaN        0    SE.PRM.ENRR
161  {'id': 'SE.PRM.ENRR', 'value': 'School enrollment, primary (% gross)'}  {'id': 'XD', 'value': 'High income'}             NaN  2024  99.806000   NaN         NaN        0    SE.PRM.ENRR
162  {'id': 'SE.PRM.ENRR', 'value': 'School enrollment, primary (% gross)'}  {'id': 'XD', 'value': 'High income'}             NaN  2023  99.709068   NaN         NaN        0    SE.PRM.ENRR
163  {'id': 'SE.PRM.ENRR', 'value': 'School enrollment, primary (% gross)'}  {'id': 'XD', 'value': 'High income'}             NaN  2022  99.637032   NaN         NaN        0    SE.PRM.ENRR


In [36]:
education_clean = education_clean[education_clean["countryiso3code"].notna()].reset_index(drop=True)

dups_combo = education_clean.duplicated(subset=["countryiso3code", "indicator_code", "date"]).sum()
print(f"Duplicate (country + indicator + date): {dups_combo}")
print(f"Shape after dropping aggregates: {education_clean.shape}")

Duplicate (country + indicator + date): 0
Shape after dropping aggregates: (2850, 9)


In [37]:
def parse_dict_col(val, key="value"):
    try:
        return ast.literal_eval(val).get(key, None)
    except:
        return None

    
for col in ["indicator", "country"]:
    education_clean[col] = education_clean[col].apply(parse_dict_col)

education_clean.drop(columns=["unit", "obs_status"], inplace=True)

print(education_clean.head(5).to_string())
print(f"\nShape: {education_clean.shape}")
print(f"\nRemaining nulls:")
print(education_clean.isnull().sum())

                              indicator                      country countryiso3code  date       value  decimal indicator_code
0  School enrollment, primary (% gross)  Africa Eastern and Southern             AFE  2025         NaN        0    SE.PRM.ENRR
1  School enrollment, primary (% gross)  Africa Eastern and Southern             AFE  2024  101.309242        0    SE.PRM.ENRR
2  School enrollment, primary (% gross)  Africa Eastern and Southern             AFE  2023  100.857590        0    SE.PRM.ENRR
3  School enrollment, primary (% gross)  Africa Eastern and Southern             AFE  2022  100.508301        0    SE.PRM.ENRR
4  School enrollment, primary (% gross)  Africa Eastern and Southern             AFE  2021  103.366081        0    SE.PRM.ENRR

Shape: (2850, 7)

Remaining nulls:
indicator            0
country              0
countryiso3code      0
date                 0
value              971
decimal              0
indicator_code       0
dtype: int64


In [38]:
education_clean.drop(columns=["decimal"], inplace=True)

real_country_codes = pd.read_csv("../cleaned/countries_clean.csv")["country_code_iso3"].tolist()
education_clean = education_clean[education_clean["countryiso3code"].isin(real_country_codes)].reset_index(drop=True)

print(f"Final shape: {education_clean.shape}")
print(f"\nRemaining nulls:")
print(education_clean.isnull().sum())
print(f"\nDate range: {education_clean['date'].min()} to {education_clean['date'].max()}")
print(f"\nUnique countries: {education_clean['countryiso3code'].nunique()}")
print(f"Unique indicators: {education_clean['indicator'].unique()}")

Final shape: (1500, 6)

Remaining nulls:
indicator            0
country              0
countryiso3code      0
date                 0
value              731
indicator_code       0
dtype: int64

Date range: 2015 to 2025

Unique countries: 50
Unique indicators: ['School enrollment, primary (% gross)'
 'School enrollment, secondary (% gross)'
 'Literacy rate, adult total (% of people ages 15 and above)']


In [39]:
education_clean.to_csv("../cleaned/education_clean.csv", index=False)
print("Saved: ../cleaned/education_clean.csv")

Saved: ../cleaned/education_clean.csv
